# BiLSTM IELTS Multi-Trait Scoring — Essay + Qwen-style Discourse Features

This notebook uses:
- `train.csv`, `val.csv`, `test.csv` from `/content`
- essay text → trainable embeddings → BiLSTM encoder
- Qwen-style handcrafted discourse features extracted from `prompt` + `essay`
- trait-specific discourse feature groups for `TR`, `CC`, `LR`, `GRA`
- four trait-specific prediction stacks
- MSE loss on scaled trait scores
- QWK/MAE metrics + derived Overall QWK


In [1]:
# =========================
# IMPORTS
# =========================

import os
import re
import math
import random
from dataclasses import dataclass
from collections import Counter, defaultdict
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import cohen_kappa_score, mean_absolute_error

try:
    from IPython.display import display
except Exception:
    display = print


In [2]:
# =========================
# REPRODUCIBILITY
# =========================

SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)


In [3]:
# =========================
# CONFIG - ESSAY + QWEN-STYLE DISCOURSE FEATURES
# =========================

RNN_OUTPUT_DIR = "/content/ielts_lstm_essay_plus_qwen_discourse_trait_stacks"
os.makedirs(RNN_OUTPUT_DIR, exist_ok=True)

TRAIT_COLS = ["TR", "CC", "LR", "GRA"]

MIN_BAND = 0.0
MAX_BAND = 9.0

# Text / vocab config
MAX_VOCAB_SIZE = 50000
MAX_LEN = 900
MIN_FREQ = 2

# Model config
EMBED_DIM = 300
RNN_HIDDEN = 384
RNN_LAYERS = 2
BIDIRECTIONAL = True

STACK_HIDDEN = 512
DISCOURSE_HIDDEN = 128
DROPOUT = 0.35

# Training config
BATCH_SIZE = 32
EPOCHS = 80
LR = 5e-4
WEIGHT_DECAY = 5e-5
MAX_GRAD_NORM = 1.0

# Early stopping config
EARLY_STOPPING_PATIENCE = 8
MIN_DELTA = 1e-4

# Keep False for clean multi-trait baseline
USE_OPTIONAL_OVERALL_CONSISTENCY_LOSS = False
OVERALL_CONSISTENCY_WEIGHT = 0.0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)


DEVICE: cuda


In [4]:
# =========================
# LOAD PRE-SPLIT CSV FILES
# Upload train.csv, val.csv, test.csv to /content first.
# =========================

TRAIN_PATH = "/content/train.csv"
VAL_PATH   = "/content/val.csv"
TEST_PATH  = "/content/test.csv"

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

print("\nTrain columns:")
print(train_df.columns.tolist())

display(train_df.head())


Train: (7743, 17)
Val: (969, 17)
Test: (970, 17)

Train columns:
['prompt', 'essay', 'evaluation', 'essay_len', 'TR', 'CC', 'LR', 'GRA', 'n_found', 'parse_quality', 'overall_raw', 'band', 'prompt_relevance', 'lexical_diversity', 'readability_score', 'target_text', 'source']


,prompt,essay,evaluation,essay_len,TR,CC,LR,GRA,n_found,parse_quality,overall_raw,band,prompt_relevance,lexical_diversity,readability_score,target_text,source
0,Some employers believe that job applicants’ so...,There has been much discussion revolving aroun...,Task Achievement:\r\n\r\n- The candidate has e...,273,7.5,7.0,7.0,7.0,4,good,7.125,7.0,0.592603,0.581818,40.754804,Analysis: This essay has a lexical diversity o...,hf
1,Some people believe that teenagers should be r...,A nation is known as a vast garden and childre...,Task Achievement:\r\nThe essay effectively add...,324,4.0,4.0,3.0,3.0,4,good,3.500,3.5,0.600165,0.563077,35.677525,Analysis: This essay has a lexical diversity o...,hf
2,some people say that economic growth is the on...,"In the modern world ,a burning issue arises th...",Task Achievement:\r\n\r\nThe essay adequately ...,348,5.5,5.0,5.0,5.0,4,good,5.125,5.0,0.746589,0.452450,40.414844,Analysis: This essay has a lexical diversity o...,hf
3,Some believe that people should make efforts t...,The issue of climate change was always debatab...,Task Achievement:\r\n- The candidate has adequ...,307,7.0,7.0,6.0,6.0,4,good,6.500,6.5,0.628859,0.629870,39.378580,Analysis: This essay has a lexical diversity o...,hf
4,Nowadays families move to different countries ...,Immigrating to other nations for business purp...,Task Achievement:\r\nThe candidate has effecti...,311,7.5,8.0,7.0,7.5,4,good,7.500,7.5,0.586910,0.612179,40.385892,Analysis: This essay has a lexical diversity o...,hf


In [5]:
# =========================
# BASIC COLUMN CHECK
# Qwen-style discourse features use prompt + essay.
# If prompt is absent, this cell creates an empty prompt column,
# but using a real prompt column is strongly recommended.
# =========================

if "prompt" not in train_df.columns:
    print("[WARN] Column 'prompt' not found. Creating empty prompt for all splits.")
    print("[WARN] Prompt-overlap features will become weak/zero.")
    for df_ in [train_df, val_df, test_df]:
        df_["prompt"] = ""

required_cols = ["prompt", "essay", "TR", "CC", "LR", "GRA"]

for name, df_ in [
    ("train_df", train_df),
    ("val_df", val_df),
    ("test_df", test_df),
]:
    missing = [c for c in required_cols if c not in df_.columns]

    if missing:
        raise ValueError(f"{name} is missing columns: {missing}")

    df_["prompt"] = df_["prompt"].fillna("").astype(str)
    df_["essay"] = df_["essay"].fillna("").astype(str)

    for col in TRAIT_COLS:
        df_[col] = df_[col].astype(float)

    if "Overall" in df_.columns:
        df_["Overall"] = df_["Overall"].astype(float)

print("All datasets are ready.")


All datasets are ready.


In [6]:
# =========================
# QWEN-STYLE DISCOURSE FEATURE EXTRACTOR
# No numeric-column auto-detection is used here.
# Features are extracted only from prompt + essay.
# =========================

STOPWORDS = {
    "a","an","the","and","or","but","if","while","although","because","as","of","to","in","on","for",
    "with","without","by","from","at","into","onto","about","over","under","between","among","through",
    "is","am","are","was","were","be","been","being","do","does","did","done","have","has","had",
    "this","that","these","those","it","its","they","them","their","we","us","our","you","your","i","me","my",
    "he","him","his","she","her","hers","not","no","so","than","then","there","here","very","can","could",
    "may","might","must","should","would","will","just","also","too","such"
}

CONNECTIVES = {
    "additive": {
        "also", "moreover", "furthermore", "besides", "additionally", "in addition", "similarly",
        "likewise", "not only", "as well as"
    },
    "contrastive": {
        "however", "nevertheless", "nonetheless", "although", "though", "even though", "whereas",
        "while", "in contrast", "on the other hand", "but", "yet", "despite", "in spite of"
    },
    "causal": {
        "because", "since", "as a result", "therefore", "thus", "hence", "consequently", "so",
        "due to", "owing to", "for this reason", "leads to", "result in", "results in"
    },
    "temporal": {
        "first", "firstly", "second", "secondly", "third", "thirdly", "finally", "lastly",
        "then", "after", "before", "when", "while", "during", "meanwhile", "subsequently"
    },
    "exemplification": {
        "for example", "for instance", "such as", "including", "namely", "to illustrate", "in particular"
    },
    "conclusive": {
        "in conclusion", "to conclude", "overall", "in summary", "to sum up", "all in all"
    },
}

SUBORDINATORS = {
    "because", "although", "though", "even though", "while", "whereas", "if", "unless",
    "when", "whenever", "since", "after", "before", "as", "so that", "provided that"
}

ACADEMIC_WORDS = {
    "significant","significantly","consequence","consequences","benefit","beneficial","detrimental",
    "impact","influence","factor","factors","issue","issues","society","individual","individuals",
    "economic","economy","environment","environmental","education","educational","technology",
    "technological","government","policy","policies","evidence","research","development","develop",
    "improve","improvement","increase","decrease","major","minor","approximately","appropriate",
    "essential","effective","efficient","solution","solutions","advantage","disadvantage",
}

THESIS_MARKERS = {
    "i believe", "i think", "in my opinion", "from my perspective", "this essay will",
    "i agree", "i disagree", "i strongly believe", "it is argued that"
}

POSITION_MARKERS = {
    "agree", "disagree", "support", "oppose", "believe", "argue", "view", "opinion", "perspective"
}

MODALS = {"can", "could", "may", "might", "must", "should", "would", "will", "ought"}

FIRST_PERSON = {"i", "me", "my", "mine", "we", "us", "our", "ours"}

VOWELS = set("aeiouy")


def simple_word_tokenize(text: str) -> List[str]:
    text = str(text).lower()
    return re.findall(r"[a-z]+(?:'[a-z]+)?|[0-9]+", text)


def simple_sentence_split(text: str) -> List[str]:
    text = str(text).strip()
    if not text:
        return []
    sents = re.split(r"(?<=[.!?])\s+", text)
    return [s.strip() for s in sents if s.strip()]


def simple_paragraph_split(text: str) -> List[str]:
    text = str(text)
    paras = re.split(r"\n\s*\n+", text.strip())
    paras = [p.strip() for p in paras if p.strip()]
    if not paras and text.strip():
        paras = [text.strip()]
    return paras


def safe_div(a, b, default=0.0):
    return float(a) / float(b) if b else default


def content_words(tokens: List[str]) -> List[str]:
    return [t for t in tokens if t not in STOPWORDS and len(t) > 2]


def rough_stem(word: str) -> str:
    for suf in ["ing", "edly", "edly", "ed", "ies", "s"]:
        if word.endswith(suf) and len(word) > len(suf) + 3:
            if suf == "ies":
                return word[:-3] + "y"
            return word[:-len(suf)]
    return word


def ngrams(tokens: List[str], n: int) -> set:
    if len(tokens) < n:
        return set()
    return set(tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1))


def count_phrase_set(text_lower: str, phrase_set: set) -> int:
    count = 0
    for phrase in phrase_set:
        pattern = r"\b" + re.escape(phrase.lower()) + r"\b"
        count += len(re.findall(pattern, text_lower))
    return count


def count_connective_categories(text_lower: str) -> Dict[str, int]:
    out = {}
    for cat, phrases in CONNECTIVES.items():
        out[cat] = count_phrase_set(text_lower, phrases)
    return out


def adjacent_sentence_overlap(sentences: List[str]) -> Tuple[float, float, float]:
    overlaps = []
    for a, b in zip(sentences[:-1], sentences[1:]):
        aw = set(content_words(simple_word_tokenize(a)))
        bw = set(content_words(simple_word_tokenize(b)))
        if not aw or not bw:
            overlaps.append(0.0)
        else:
            overlaps.append(len(aw & bw) / max(1, min(len(aw), len(bw))))

    if not overlaps:
        return 0.0, 0.0, 0.0

    arr = np.asarray(overlaps, dtype=np.float32)
    return float(arr.mean()), float(arr.std()), float(arr.max())


def paragraph_balance(paragraphs: List[str]) -> float:
    lengths = [len(simple_word_tokenize(p)) for p in paragraphs]
    if len(lengths) <= 1:
        return 1.0

    mean_len = np.mean(lengths)
    if mean_len <= 0:
        return 0.0

    cv = np.std(lengths) / mean_len
    return float(1.0 / (1.0 + cv))


def lexical_chain_proxy(tokens: List[str]) -> Dict[str, float]:
    stems = [rough_stem(t) for t in content_words(tokens)]
    counts = Counter(stems)

    chains = [c for c in counts.values() if c >= 2]
    large = [c for c in counts.values() if c >= 3]

    chain_count = len(chains)
    lexical_chain_coverage = safe_div(sum(chains), len(tokens))

    # approximate varied chains: stems that appear multiple times but not excessively
    varied = [c for c in counts.values() if 3 <= c <= 8]

    return {
        "lexical_chain_count": float(chain_count),
        "avg_lexical_chain_size": float(np.mean(chains)) if chains else 0.0,
        "num_large_lexical_chains": float(len(large)),
        "num_large_varied_lexical_chains": float(len(varied)),
        "pct_large_lexical_chains": safe_div(len(large), max(1, chain_count)),
        "pct_large_varied_lexical_chains": safe_div(len(varied), max(1, chain_count)),
        "lexical_chain_coverage": lexical_chain_coverage,
        "lexical_chain_variety": safe_div(chain_count, max(1, len(set(stems)))),
    }


def count_syllable_proxy(word: str) -> int:
    word = word.lower()
    if not word:
        return 0

    count = 0
    prev_vowel = False

    for ch in word:
        is_vowel = ch in VOWELS
        if is_vowel and not prev_vowel:
            count += 1
        prev_vowel = is_vowel

    if word.endswith("e") and count > 1:
        count -= 1

    return max(1, count)


def readability_features(tokens: List[str], sentence_count: int) -> Dict[str, float]:
    word_count = len(tokens)
    syllables = sum(count_syllable_proxy(t) for t in tokens)

    syllable_per_word = safe_div(syllables, word_count)
    words_per_sentence = safe_div(word_count, sentence_count)

    # proxy formulas
    fre = 206.835 - 1.015 * words_per_sentence - 84.6 * syllable_per_word
    fk_grade = 0.39 * words_per_sentence + 11.8 * syllable_per_word - 15.59

    return {
        "syllable_proxy_per_word": float(syllable_per_word),
        "flesch_reading_ease_proxy": float(fre),
        "flesch_kincaid_grade_proxy": float(fk_grade),
    }


def extract_discourse_features(prompt: str, essay: str) -> Dict[str, float]:
    prompt = "" if pd.isna(prompt) else str(prompt)
    essay = "" if pd.isna(essay) else str(essay)

    text_lower = essay.lower()
    prompt_lower = prompt.lower()

    tokens = simple_word_tokenize(essay)
    prompt_tokens = simple_word_tokenize(prompt)
    sentences = simple_sentence_split(essay)
    paragraphs = simple_paragraph_split(essay)

    word_count = len(tokens)
    unique_words = len(set(tokens))
    sentence_count = len(sentences)
    paragraph_count = len(paragraphs)

    sentence_lens = [len(simple_word_tokenize(s)) for s in sentences]
    avg_sentence_len = float(np.mean(sentence_lens)) if sentence_lens else 0.0
    sentence_len_std = float(np.std(sentence_lens)) if sentence_lens else 0.0

    prompt_content = set(content_words(prompt_tokens))
    essay_content = set(content_words(tokens))

    prompt_overlap = safe_div(len(prompt_content & essay_content), len(prompt_content))

    prompt_bigrams = ngrams(content_words(prompt_tokens), 2)
    essay_bigrams = ngrams(content_words(tokens), 2)
    prompt_bigram_overlap = safe_div(len(prompt_bigrams & essay_bigrams), len(prompt_bigrams))

    conn_counts = count_connective_categories(text_lower)
    connective_count = sum(conn_counts.values())

    adj_mean, adj_std, adj_max = adjacent_sentence_overlap(sentences)
    lex_chain = lexical_chain_proxy(tokens)
    read = readability_features(tokens, sentence_count)

    punctuation_error_proxy = len(re.findall(r"\s+[,.!?;:]", essay)) + len(re.findall(r"[,.!?;:]{2,}", essay))
    question_exclaim = essay.count("?") + essay.count("!")

    bigram_div = safe_div(len(ngrams(tokens, 2)), max(1, word_count - 1))
    trigram_div = safe_div(len(ngrams(tokens, 3)), max(1, word_count - 2))

    subordinator_count = count_phrase_set(text_lower, SUBORDINATORS)

    complex_sentence_count = 0
    for s in sentences:
        s_lower = s.lower()
        if "," in s_lower or any(re.search(r"\b" + re.escape(sub) + r"\b", s_lower) for sub in SUBORDINATORS):
            complex_sentence_count += 1

    very_long_sentence_count = sum(1 for x in sentence_lens if x >= 30)
    very_short_sentence_count = sum(1 for x in sentence_lens if x <= 7)

    avg_word_len = float(np.mean([len(t) for t in tokens])) if tokens else 0.0

    feats = {
        "word_count": float(word_count),
        "sentence_count": float(sentence_count),
        "paragraph_count": float(paragraph_count),
        "prompt_word_count": float(len(prompt_tokens)),

        "prompt_overlap": float(prompt_overlap),
        "prompt_bigram_overlap": float(prompt_bigram_overlap),

        "avg_sentence_len": float(avg_sentence_len),
        "sentence_len_std": float(sentence_len_std),
        "paragraph_balance": paragraph_balance(paragraphs),

        "connective_count": float(connective_count),
        "connective_per_sentence": safe_div(connective_count, sentence_count),

        "additive_connective_count": float(conn_counts.get("additive", 0)),
        "contrastive_connective_count": float(conn_counts.get("contrastive", 0)),
        "causal_connective_count": float(conn_counts.get("causal", 0)),
        "temporal_connective_count": float(conn_counts.get("temporal", 0)),
        "exemplification_connective_count": float(conn_counts.get("exemplification", 0)),
        "conclusive_connective_count": float(conn_counts.get("conclusive", 0)),

        "unique_word_count": float(unique_words),
        "avg_word_len": avg_word_len,
        "type_token_ratio": safe_div(unique_words, word_count),
        "root_ttr": safe_div(unique_words, math.sqrt(word_count)) if word_count else 0.0,
        "hapax_ratio": safe_div(sum(1 for _, c in Counter(tokens).items() if c == 1), word_count),
        "long_word_ratio": safe_div(sum(1 for t in tokens if len(t) >= 7), word_count),

        "bigram_diversity": float(bigram_div),
        "trigram_diversity": float(trigram_div),
        "academic_word_ratio": safe_div(sum(1 for t in tokens if t in ACADEMIC_WORDS), word_count),

        "modal_ratio": safe_div(sum(1 for t in tokens if t in MODALS), word_count),
        "first_person_ratio": safe_div(sum(1 for t in tokens if t in FIRST_PERSON), word_count),
        "thesis_marker_count": float(count_phrase_set(text_lower, THESIS_MARKERS)),
        "position_marker_count": float(sum(1 for t in tokens if t in POSITION_MARKERS)),

        "comma_per_sentence": safe_div(essay.count(","), sentence_count),
        "semicolon_colon_per_sentence": safe_div(essay.count(";") + essay.count(":"), sentence_count),
        "subordinator_count": float(subordinator_count),
        "subordinator_ratio": safe_div(subordinator_count, sentence_count),
        "complex_sentence_ratio": safe_div(complex_sentence_count, sentence_count),
        "very_long_sentence_ratio": safe_div(very_long_sentence_count, sentence_count),
        "very_short_sentence_ratio": safe_div(very_short_sentence_count, sentence_count),
        "punctuation_error_proxy": float(punctuation_error_proxy),
        "question_exclaim_per_sentence": safe_div(question_exclaim, sentence_count),
    }

    feats.update(lex_chain)
    feats.update(read)

    return feats


In [7]:
# =========================
# QWEN-STYLE FEATURE GROUPS
# Each trait receives its own discourse feature subset.
# =========================

FEATURE_GROUPS = {
    "TR": [
        "word_count",
        "sentence_count",
        "prompt_word_count",
        "prompt_overlap",
        "prompt_bigram_overlap",
        "modal_ratio",
        "first_person_ratio",
        "thesis_marker_count",
        "position_marker_count",
    ],

    "CC": [
        "paragraph_count",
        "sentence_count",
        "avg_sentence_len",
        "sentence_len_std",
        "adjacent_sentence_overlap_mean",
        "adjacent_sentence_overlap_std",
        "adjacent_sentence_overlap_max",
        "paragraph_balance",
        "connective_count",
        "connective_per_sentence",
        "additive_connective_count",
        "contrastive_connective_count",
        "causal_connective_count",
        "temporal_connective_count",
        "exemplification_connective_count",
        "conclusive_connective_count",
        "lexical_chain_count",
        "avg_lexical_chain_size",
        "num_large_lexical_chains",
        "num_large_varied_lexical_chains",
        "pct_large_lexical_chains",
        "pct_large_varied_lexical_chains",
        "lexical_chain_coverage",
    ],

    "LR": [
        "word_count",
        "unique_word_count",
        "avg_word_len",
        "type_token_ratio",
        "root_ttr",
        "hapax_ratio",
        "long_word_ratio",
        "bigram_diversity",
        "trigram_diversity",
        "academic_word_ratio",
        "lexical_chain_count",
        "lexical_chain_variety",
        "lexical_chain_coverage",
    ],

    "GRA": [
        "sentence_count",
        "avg_sentence_len",
        "sentence_len_std",
        "comma_per_sentence",
        "semicolon_colon_per_sentence",
        "subordinator_count",
        "subordinator_ratio",
        "complex_sentence_ratio",
        "very_long_sentence_ratio",
        "very_short_sentence_ratio",
        "punctuation_error_proxy",
        "question_exclaim_per_sentence",
        "syllable_proxy_per_word",
        "flesch_reading_ease_proxy",
        "flesch_kincaid_grade_proxy",
    ],
}

# Add missing overlap feature names to extractor output through compatibility mapping.
# The extractor computes adjacent overlap below via helper, so patch it here by wrapping original.
_original_extract_discourse_features = extract_discourse_features

def extract_discourse_features(prompt: str, essay: str) -> Dict[str, float]:
    feats = _original_extract_discourse_features(prompt, essay)
    sentences = simple_sentence_split("" if pd.isna(essay) else str(essay))
    adj_mean, adj_std, adj_max = adjacent_sentence_overlap(sentences)
    feats["adjacent_sentence_overlap_mean"] = adj_mean
    feats["adjacent_sentence_overlap_std"] = adj_std
    feats["adjacent_sentence_overlap_max"] = adj_max
    return feats

ALL_FEATURES = sorted(set(sum(FEATURE_GROUPS.values(), [])))

FEATURE_IDXS = {
    trait: [ALL_FEATURES.index(f) for f in FEATURE_GROUPS[trait]]
    for trait in TRAIT_COLS
}

FEATURE_DIMS = {
    trait: len(FEATURE_IDXS[trait])
    for trait in TRAIT_COLS
}

print("Total discourse features:", len(ALL_FEATURES))

for trait in TRAIT_COLS:
    print(trait, FEATURE_DIMS[trait], [ALL_FEATURES[i] for i in FEATURE_IDXS[trait]])


Total discourse features: 53
TR 9 ['word_count', 'sentence_count', 'prompt_word_count', 'prompt_overlap', 'prompt_bigram_overlap', 'modal_ratio', 'first_person_ratio', 'thesis_marker_count', 'position_marker_count']
CC 23 ['paragraph_count', 'sentence_count', 'avg_sentence_len', 'sentence_len_std', 'adjacent_sentence_overlap_mean', 'adjacent_sentence_overlap_std', 'adjacent_sentence_overlap_max', 'paragraph_balance', 'connective_count', 'connective_per_sentence', 'additive_connective_count', 'contrastive_connective_count', 'causal_connective_count', 'temporal_connective_count', 'exemplification_connective_count', 'conclusive_connective_count', 'lexical_chain_count', 'avg_lexical_chain_size', 'num_large_lexical_chains', 'num_large_varied_lexical_chains', 'pct_large_lexical_chains', 'pct_large_varied_lexical_chains', 'lexical_chain_coverage']
LR 13 ['word_count', 'unique_word_count', 'avg_word_len', 'type_token_ratio', 'root_ttr', 'hapax_ratio', 'long_word_ratio', 'bigram_diversity', 'tr

In [8]:
# =========================
# FEATURE NORMALIZER
# Fit statistics on train only, then transform train/val/test.
# =========================

class FeatureNormalizer:
    def __init__(self, feature_names):
        self.feature_names = list(feature_names)
        self.mean = None
        self.std = None

    def _raw_row(self, prompt, essay):
        d = extract_discourse_features(prompt, essay)
        return np.asarray([d.get(f, 0.0) for f in self.feature_names], dtype=np.float32)

    def fit(self, df):
        feats = []

        for _, row in tqdm(df.iterrows(), total=len(df), desc="Extract train discourse features"):
            feats.append(self._raw_row(row["prompt"], row["essay"]))

        arr = np.asarray(feats, dtype=np.float32)

        arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)

        self.mean = arr.mean(axis=0)
        self.std = arr.std(axis=0)
        self.std[self.std < 1e-6] = 1.0

        return self

    def transform_row(self, prompt, essay):
        arr = self._raw_row(prompt, essay)
        arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
        return ((arr - self.mean) / self.std).astype(np.float32)

    def transform_df(self, df, desc="Extract discourse features"):
        rows = []

        for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
            rows.append(self.transform_row(row["prompt"], row["essay"]))

        return np.asarray(rows, dtype=np.float32)


feature_normalizer = FeatureNormalizer(ALL_FEATURES).fit(train_df)

train_discourse = feature_normalizer.transform_df(train_df, desc="Transform train discourse features")
val_discourse   = feature_normalizer.transform_df(val_df, desc="Transform val discourse features")
test_discourse  = feature_normalizer.transform_df(test_df, desc="Transform test discourse features")

DISC_FEAT_DIM = len(ALL_FEATURES)

print("DISC_FEAT_DIM:", DISC_FEAT_DIM)
print("train_discourse:", train_discourse.shape)
print("val_discourse:", val_discourse.shape)
print("test_discourse:", test_discourse.shape)


Extract train discourse features:   0%|          | 0/7743 [00:00<?, ?it/s]

Transform train discourse features:   0%|          | 0/7743 [00:00<?, ?it/s]

Transform val discourse features:   0%|          | 0/969 [00:00<?, ?it/s]

Transform test discourse features:   0%|          | 0/970 [00:00<?, ?it/s]

DISC_FEAT_DIM: 53
train_discourse: (7743, 53)
val_discourse: (969, 53)
test_discourse: (970, 53)


In [9]:
# =========================
# METRIC FUNCTIONS
# Includes derived Overall QWK
# =========================

def band_to_scaled(x):
    x = np.asarray(x, dtype=np.float32)
    return (x - MIN_BAND) / (MAX_BAND - MIN_BAND)


def scaled_to_band(x):
    x = np.asarray(x, dtype=np.float32)
    band = x * (MAX_BAND - MIN_BAND) + MIN_BAND
    band = np.clip(band, MIN_BAND, MAX_BAND)
    band = np.round(band * 2) / 2
    return band


def qwk_band(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)

    valid = ~np.isnan(y_true) & ~np.isnan(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]

    if len(y_true) == 0:
        return np.nan

    y_true_int = (y_true * 2).astype(int)
    y_pred_int = (y_pred * 2).astype(int)

    return cohen_kappa_score(
        y_true_int,
        y_pred_int,
        weights="quadratic",
    )


def compute_metrics(y_true_scaled, y_pred_scaled, overall_true=None):
    y_true_band = scaled_to_band(y_true_scaled)
    y_pred_band = scaled_to_band(y_pred_scaled)

    metrics = {}

    trait_qwks = []
    trait_maes = []

    for i, trait in enumerate(TRAIT_COLS):
        qwk = qwk_band(y_true_band[:, i], y_pred_band[:, i])
        mae = mean_absolute_error(y_true_band[:, i], y_pred_band[:, i])

        metrics[f"{trait}_qwk"] = qwk
        metrics[f"{trait}_mae"] = mae

        trait_qwks.append(qwk)
        trait_maes.append(mae)

    metrics["mean_trait_qwk"] = float(np.nanmean(trait_qwks))
    metrics["mean_trait_mae"] = float(np.nanmean(trait_maes))

    derived_overall_pred = np.mean(y_pred_band, axis=1)
    derived_overall_pred = np.round(derived_overall_pred * 2) / 2
    derived_overall_pred = np.clip(derived_overall_pred, MIN_BAND, MAX_BAND)

    derived_overall_gold_from_traits = np.mean(y_true_band, axis=1)
    derived_overall_gold_from_traits = np.round(derived_overall_gold_from_traits * 2) / 2
    derived_overall_gold_from_traits = np.clip(derived_overall_gold_from_traits, MIN_BAND, MAX_BAND)

    metrics["derived_overall_from_traits_qwk"] = qwk_band(
        derived_overall_gold_from_traits,
        derived_overall_pred,
    )

    metrics["derived_overall_from_traits_mae"] = mean_absolute_error(
        derived_overall_gold_from_traits,
        derived_overall_pred,
    )

    if overall_true is not None:
        overall_true = np.asarray(overall_true, dtype=np.float32)
        valid = ~np.isnan(overall_true)

        if valid.any():
            metrics["derived_overall_qwk"] = qwk_band(
                overall_true[valid],
                derived_overall_pred[valid],
            )

            metrics["derived_overall_mae"] = mean_absolute_error(
                overall_true[valid],
                derived_overall_pred[valid],
            )

    return metrics, y_true_band, y_pred_band, derived_overall_pred


In [10]:
# =========================
# TOKENIZER + VOCAB
# =========================

def simple_tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9.,!?;:'\"()\- ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.split()


class Vocab:
    def __init__(self, max_size=30000, min_freq=2):
        self.max_size = max_size
        self.min_freq = min_freq

        self.pad_token = "<PAD>"
        self.unk_token = "<UNK>"

        self.stoi = {
            self.pad_token: 0,
            self.unk_token: 1,
        }
        self.itos = [self.pad_token, self.unk_token]

    def fit(self, texts):
        counter = Counter()

        for text in texts:
            counter.update(simple_tokenize(text))

        words = [
            word for word, freq in counter.most_common()
            if freq >= self.min_freq
        ]

        words = words[: self.max_size - len(self.itos)]

        for word in words:
            if word not in self.stoi:
                self.stoi[word] = len(self.itos)
                self.itos.append(word)

        print(f"Vocab size: {len(self.itos)}")

    def encode(self, text, max_len):
        tokens = simple_tokenize(text)

        ids = [
            self.stoi.get(tok, self.stoi[self.unk_token])
            for tok in tokens
        ]

        if len(ids) > max_len:
            ids = ids[:max_len]

        if len(ids) == 0:
            ids = [self.stoi[self.unk_token]]

        return ids


vocab = Vocab(max_size=MAX_VOCAB_SIZE, min_freq=MIN_FREQ)
vocab.fit(train_df["essay"].astype(str).tolist())


Vocab size: 27362


In [11]:
# =========================
# DATASET + COLLATOR
# Uses precomputed discourse features.
# =========================

class IELTSEssayDiscourseDataset(Dataset):
    def __init__(self, df, vocab, discourse_array, max_len=600):
        self.df = df.reset_index(drop=True)
        self.vocab = vocab
        self.discourse_array = discourse_array.astype(np.float32)
        self.max_len = max_len

        assert len(self.df) == len(self.discourse_array)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        input_ids = self.vocab.encode(row["essay"], self.max_len)
        discourse_features = self.discourse_array[idx]

        labels_band = row[TRAIT_COLS].astype(float).values.astype(np.float32)
        labels_scaled = band_to_scaled(labels_band).astype(np.float32)

        if "Overall" in self.df.columns and not pd.isna(row.get("Overall", np.nan)):
            overall_band = np.float32(row["Overall"])
        else:
            overall_band = np.float32(np.nan)

        return {
            "input_ids": input_ids,
            "discourse_features": discourse_features,
            "labels": labels_scaled,
            "overall_band": overall_band,
        }


@dataclass
class RNNDataCollator:
    pad_id: int = 0

    def __call__(self, batch):
        lengths = [len(x["input_ids"]) for x in batch]
        max_len = max(lengths)

        input_ids = []
        attention_mask = []

        for item in batch:
            ids = item["input_ids"]
            pad_len = max_len - len(ids)

            input_ids.append(ids + [self.pad_id] * pad_len)
            attention_mask.append([1] * len(ids) + [0] * pad_len)

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "lengths": torch.tensor(lengths, dtype=torch.long),

            "discourse_features": torch.tensor(
                np.stack([x["discourse_features"] for x in batch]),
                dtype=torch.float32,
            ),

            "labels": torch.tensor(
                np.stack([x["labels"] for x in batch]),
                dtype=torch.float32,
            ),

            "overall_band": torch.tensor(
                [x["overall_band"] for x in batch],
                dtype=torch.float32,
            ),
        }


train_dataset = IELTSEssayDiscourseDataset(
    train_df,
    vocab,
    discourse_array=train_discourse,
    max_len=MAX_LEN,
)

val_dataset = IELTSEssayDiscourseDataset(
    val_df,
    vocab,
    discourse_array=val_discourse,
    max_len=MAX_LEN,
)

test_dataset = IELTSEssayDiscourseDataset(
    test_df,
    vocab,
    discourse_array=test_discourse,
    max_len=MAX_LEN,
)

collator = RNNDataCollator(pad_id=vocab.stoi["<PAD>"])

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collator,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
)


In [12]:
# =========================
# MODEL
# BiLSTM essay encoder + trait-specific discourse feature groups
# =========================

class TraitStack(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, dropout=0.3):
        super().__init__()

        self.net = nn.Sequential(
            nn.LayerNorm(input_dim),

            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim // 2, hidden_dim // 4),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim // 4, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


class EssayLSTMDiscourseTraitScorer(nn.Module):
    def __init__(
        self,
        vocab_size,
        feature_dims,
        feature_idxs,
        embed_dim=200,
        rnn_hidden=256,
        rnn_layers=2,
        bidirectional=True,
        discourse_hidden=128,
        stack_hidden=256,
        dropout=0.3,
        pad_id=0,
    ):
        super().__init__()

        self.feature_idxs = feature_idxs

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_id,
        )

        self.embedding_dropout = nn.Dropout(dropout)

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=rnn_hidden,
            num_layers=rnn_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if rnn_layers > 1 else 0.0,
        )

        direction_factor = 2 if bidirectional else 1
        encoder_dim = rnn_hidden * direction_factor
        essay_repr_dim = encoder_dim * 3

        self.essay_norm = nn.LayerNorm(essay_repr_dim)

        self.feature_projectors = nn.ModuleDict({
            trait: nn.Sequential(
                nn.LayerNorm(feature_dims[trait]),
                nn.Linear(feature_dims[trait], discourse_hidden),
                nn.GELU(),
                nn.Dropout(dropout),
            )
            for trait in TRAIT_COLS
        })

        fusion_dim = essay_repr_dim + discourse_hidden

        self.stacks = nn.ModuleDict({
            trait: TraitStack(
                input_dim=fusion_dim,
                hidden_dim=stack_hidden,
                dropout=dropout,
            )
            for trait in TRAIT_COLS
        })

    def masked_mean_pooling(self, outputs, mask):
        mask = mask.unsqueeze(-1).float()
        summed = (outputs * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1.0)
        return summed / counts

    def masked_max_pooling(self, outputs, mask):
        mask = mask.unsqueeze(-1).bool()
        outputs = outputs.masked_fill(~mask, -1e9)
        return outputs.max(dim=1).values

    def encode_essay(self, input_ids, attention_mask, lengths):
        embedded = self.embedding(input_ids)
        embedded = self.embedding_dropout(embedded)

        lengths_cpu = lengths.detach().cpu()

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths_cpu,
            batch_first=True,
            enforce_sorted=False,
        )

        packed_outputs, (h_n, c_n) = self.lstm(packed)

        outputs, _ = nn.utils.rnn.pad_packed_sequence(
            packed_outputs,
            batch_first=True,
            total_length=input_ids.size(1),
        )

        mean_pool = self.masked_mean_pooling(outputs, attention_mask)
        max_pool = self.masked_max_pooling(outputs, attention_mask)

        if self.lstm.bidirectional:
            last_forward = h_n[-2]
            last_backward = h_n[-1]
            last_hidden = torch.cat([last_forward, last_backward], dim=-1)
        else:
            last_hidden = h_n[-1]

        essay_repr = torch.cat([mean_pool, max_pool, last_hidden], dim=-1)
        essay_repr = self.essay_norm(essay_repr)

        return essay_repr

    def forward(self, input_ids, attention_mask, lengths, discourse_features):
        essay_repr = self.encode_essay(
            input_ids=input_ids,
            attention_mask=attention_mask,
            lengths=lengths,
        )

        preds = []

        for trait in TRAIT_COLS:
            idxs = torch.tensor(
                self.feature_idxs[trait],
                dtype=torch.long,
                device=discourse_features.device,
            )

            trait_features = discourse_features.index_select(
                dim=1,
                index=idxs,
            )

            trait_feature_repr = self.feature_projectors[trait](trait_features)

            trait_repr = torch.cat(
                [essay_repr, trait_feature_repr],
                dim=-1,
            )

            pred = self.stacks[trait](trait_repr)
            preds.append(pred)

        preds = torch.stack(preds, dim=1)

        # Output range: scaled 0-1
        preds = torch.sigmoid(preds)

        return preds


model = EssayLSTMDiscourseTraitScorer(
    vocab_size=len(vocab.itos),
    feature_dims=FEATURE_DIMS,
    feature_idxs=FEATURE_IDXS,
    embed_dim=EMBED_DIM,
    rnn_hidden=RNN_HIDDEN,
    rnn_layers=RNN_LAYERS,
    bidirectional=BIDIRECTIONAL,
    discourse_hidden=DISCOURSE_HIDDEN,
    stack_hidden=STACK_HIDDEN,
    dropout=DROPOUT,
    pad_id=vocab.stoi["<PAD>"],
).to(DEVICE)

print(model)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total params: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")


EssayLSTMDiscourseTraitScorer(
  (embedding): Embedding(27362, 300, padding_idx=0)
  (embedding_dropout): Dropout(p=0.35, inplace=False)
  (lstm): LSTM(300, 384, num_layers=2, batch_first=True, dropout=0.35, bidirectional=True)
  (essay_norm): LayerNorm((2304,), eps=1e-05, elementwise_affine=True)
  (feature_projectors): ModuleDict(
    (TR): Sequential(
      (0): LayerNorm((9,), eps=1e-05, elementwise_affine=True)
      (1): Linear(in_features=9, out_features=128, bias=True)
      (2): GELU(approximate='none')
      (3): Dropout(p=0.35, inplace=False)
    )
    (CC): Sequential(
      (0): LayerNorm((23,), eps=1e-05, elementwise_affine=True)
      (1): Linear(in_features=23, out_features=128, bias=True)
      (2): GELU(approximate='none')
      (3): Dropout(p=0.35, inplace=False)
    )
    (LR): Sequential(
      (0): LayerNorm((13,), eps=1e-05, elementwise_affine=True)
      (1): Linear(in_features=13, out_features=128, bias=True)
      (2): GELU(approximate='none')
      (3): Dropo

In [13]:
# =========================
# LOSS
# =========================

def criterion_loss_lstm(pred_scaled, labels_scaled, overall_band=None):
    loss = 0.0

    for i, trait in enumerate(TRAIT_COLS):
        loss = loss + F.mse_loss(pred_scaled[:, i], labels_scaled[:, i])

    loss = loss / len(TRAIT_COLS)

    if USE_OPTIONAL_OVERALL_CONSISTENCY_LOSS and overall_band is not None:
        valid = ~torch.isnan(overall_band)

        if valid.any():
            pred_overall_scaled = pred_scaled.mean(dim=1)

            gold_overall_scaled = (
                overall_band.to(pred_scaled.device) - MIN_BAND
            ) / (MAX_BAND - MIN_BAND)

            loss = loss + OVERALL_CONSISTENCY_WEIGHT * F.mse_loss(
                pred_overall_scaled[valid],
                gold_overall_scaled[valid],
            )

    return loss


In [14]:
# =========================
# EVALUATION HELPERS
# =========================

def move_batch_to_device(batch):
    out = {}

    for k, v in batch.items():
        if isinstance(v, torch.Tensor):
            out[k] = v.to(DEVICE)
        else:
            out[k] = v

    return out


@torch.no_grad()
def predict_loader_lstm(model, loader):
    model.eval()

    all_preds = []
    all_labels = []
    all_overall = []

    for batch in tqdm(loader, desc="Predict"):
        batch = move_batch_to_device(batch)

        preds = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            lengths=batch["lengths"],
            discourse_features=batch["discourse_features"],
        )

        all_preds.append(preds.detach().cpu().numpy())
        all_labels.append(batch["labels"].detach().cpu().numpy())
        all_overall.append(batch["overall_band"].detach().cpu().numpy())

    return (
        np.concatenate(all_labels, axis=0),
        np.concatenate(all_preds, axis=0),
        np.concatenate(all_overall, axis=0),
    )


def print_metrics(metrics, title):
    print("\n" + title)
    print("=" * len(title))

    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"{k}: {v:.4f}")
        else:
            print(f"{k}: {v}")


def evaluate_lstm(model, loader, name="val"):
    y_true, y_pred, overall_true = predict_loader_lstm(model, loader)

    metrics, y_true_band, y_pred_band, overall_pred = compute_metrics(
        y_true,
        y_pred,
        overall_true,
    )

    print_metrics(metrics, f"== {name} ==")

    return metrics, y_true_band, y_pred_band, overall_pred


In [15]:
# =========================
# TRAIN LSTM ESSAY + QWEN-STYLE DISCOURSE MODEL WITH EARLY STOPPING
# =========================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2,
)

epochs_no_improve = 0
best_val_qwk = -1.0

best_path = os.path.join(RNN_OUTPUT_DIR, "best_lstm_essay_plus_qwen_discourse.pt")

for epoch in range(1, EPOCHS + 1):
    model.train()

    running_loss = 0.0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")

    for step, batch in enumerate(pbar, start=1):
        batch = move_batch_to_device(batch)

        optimizer.zero_grad(set_to_none=True)

        preds = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            lengths=batch["lengths"],
            discourse_features=batch["discourse_features"],
        )

        loss = criterion_loss_lstm(
            preds,
            batch["labels"],
            batch["overall_band"],
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            MAX_GRAD_NORM,
        )

        optimizer.step()

        running_loss += loss.item()
        avg_running_loss = running_loss / step

        pbar.set_postfix({
            "loss": f"{avg_running_loss:.6f}",
            "lr": optimizer.param_groups[0]["lr"],
        })

    avg_train_loss = running_loss / len(train_loader)

    print(f"\nEpoch {epoch} train loss: {avg_train_loss:.6f}")

    val_metrics, _, _, _ = evaluate_lstm(
        model,
        val_loader,
        name=f"val epoch {epoch}",
    )

    # Main validation metric for multi-trait scoring
    if "mean_trait_qwk" in val_metrics:
        current_qwk = val_metrics["mean_trait_qwk"]
        selected_metric_name = "mean_trait_qwk"
    elif "avg_trait_qwk" in val_metrics:
        current_qwk = val_metrics["avg_trait_qwk"]
        selected_metric_name = "avg_trait_qwk"
    elif "trait_mean_qwk" in val_metrics:
        current_qwk = val_metrics["trait_mean_qwk"]
        selected_metric_name = "trait_mean_qwk"
    else:
        qwk_keys = [
            k for k in val_metrics.keys()
            if "qwk" in k.lower()
            and "overall" not in k.lower()
        ]

        if len(qwk_keys) == 0:
            qwk_keys = [
                k for k in val_metrics.keys()
                if "qwk" in k.lower()
            ]

        if len(qwk_keys) == 0:
            raise ValueError(
                "No QWK metric found in val_metrics. "
                f"Available keys: {list(val_metrics.keys())}"
            )

        current_qwk = float(np.nanmean([val_metrics[k] for k in qwk_keys]))
        selected_metric_name = "mean_of_" + "_".join(qwk_keys)

    scheduler.step(current_qwk)

    print(f"Selected validation metric: {selected_metric_name}")
    print(f"Current val QWK: {current_qwk:.4f}")
    print(f"Best val QWK so far: {best_val_qwk:.4f}")

    if "derived_overall_qwk" in val_metrics:
        print(f"Val derived_overall_qwk: {val_metrics['derived_overall_qwk']:.4f}")

    if "derived_overall_from_traits_qwk" in val_metrics:
        print(
            "Val derived_overall_from_traits_qwk: "
            f"{val_metrics['derived_overall_from_traits_qwk']:.4f}"
        )

    improved = current_qwk > best_val_qwk + MIN_DELTA

    if improved:
        best_val_qwk = current_qwk
        epochs_no_improve = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),

                "best_val_qwk": best_val_qwk,
                "selected_metric_name": selected_metric_name,
                "best_val_metrics": val_metrics,

                "vocab_stoi": vocab.stoi,
                "vocab_itos": vocab.itos,

                "feature_mean": feature_normalizer.mean,
                "feature_std": feature_normalizer.std,

                "config": {
                    "FEATURE_MODE": "essay_plus_qwen_style_discourse",

                    "MAX_LEN": MAX_LEN,
                    "MAX_VOCAB_SIZE": MAX_VOCAB_SIZE,
                    "MIN_FREQ": MIN_FREQ,

                    "EMBED_DIM": EMBED_DIM,
                    "RNN_HIDDEN": RNN_HIDDEN,
                    "RNN_LAYERS": RNN_LAYERS,
                    "BIDIRECTIONAL": BIDIRECTIONAL,

                    "STACK_HIDDEN": STACK_HIDDEN,
                    "DISCOURSE_HIDDEN": DISCOURSE_HIDDEN,
                    "DROPOUT": DROPOUT,

                    "DISC_FEAT_DIM": DISC_FEAT_DIM,
                    "ALL_FEATURES": ALL_FEATURES,
                    "FEATURE_GROUPS": FEATURE_GROUPS,
                    "FEATURE_IDXS": FEATURE_IDXS,
                    "FEATURE_DIMS": FEATURE_DIMS,

                    "BATCH_SIZE": BATCH_SIZE,
                    "EPOCHS": EPOCHS,
                    "LR": LR,
                    "WEIGHT_DECAY": WEIGHT_DECAY,
                    "MAX_GRAD_NORM": MAX_GRAD_NORM,

                    "TRAIT_COLS": TRAIT_COLS,
                    "MIN_BAND": MIN_BAND,
                    "MAX_BAND": MAX_BAND,

                    "USE_OPTIONAL_OVERALL_CONSISTENCY_LOSS": USE_OPTIONAL_OVERALL_CONSISTENCY_LOSS,
                    "OVERALL_CONSISTENCY_WEIGHT": OVERALL_CONSISTENCY_WEIGHT,

                    "EARLY_STOPPING_PATIENCE": EARLY_STOPPING_PATIENCE,
                    "MIN_DELTA": MIN_DELTA,
                },
            },
            best_path,
        )

        print(f"[SAVE] Best model saved to: {best_path}")
        print(f"[SAVE] best_val_qwk = {best_val_qwk:.4f}")

    else:
        epochs_no_improve += 1

        print(
            f"[EARLY STOPPING] No improvement for "
            f"{epochs_no_improve}/{EARLY_STOPPING_PATIENCE} epochs."
        )

        if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
            print("[EARLY STOPPING] Triggered.")
            print(f"Stopped at epoch {epoch}.")
            break

    print("-" * 80)

print("\nTraining finished.")
print(f"Best validation QWK: {best_val_qwk:.4f}")
print(f"Best checkpoint path: {best_path}")


Epoch 1/80:   0%|          | 0/242 [00:00<?, ?it/s]


Epoch 1 train loss: 0.032195


Predict:   0%|          | 0/31 [00:00<?, ?it/s]


== val epoch 1 ==
TR_qwk: 0.1794
TR_mae: 1.0774
CC_qwk: 0.1653
CC_mae: 1.2105
LR_qwk: 0.1991
LR_mae: 1.1109
GRA_qwk: 0.2310
GRA_mae: 1.1847
mean_trait_qwk: 0.1937
mean_trait_mae: 1.1459
derived_overall_from_traits_qwk: 0.1660
derived_overall_from_traits_mae: 1.1414
Selected validation metric: mean_trait_qwk
Current val QWK: 0.1937
Best val QWK so far: -1.0000
Val derived_overall_from_traits_qwk: 0.1660
[SAVE] Best model saved to: /content/ielts_lstm_essay_plus_qwen_discourse_trait_stacks/best_lstm_essay_plus_qwen_discourse.pt
[SAVE] best_val_qwk = 0.1937
--------------------------------------------------------------------------------


Epoch 2/80:   0%|          | 0/242 [00:00<?, ?it/s]

...


== val epoch 25 ==
TR_qwk: 0.3882
TR_mae: 1.1553
CC_qwk: 0.3806
CC_mae: 1.3013
LR_qwk: 0.3944
LR_mae: 1.2301
GRA_qwk: 0.4244
GRA_mae: 1.2580
mean_trait_qwk: 0.3969
mean_trait_mae: 1.2362
derived_overall_from_traits_qwk: 0.3995
derived_overall_from_traits_mae: 1.2337
Selected validation metric: mean_trait_qwk
Current val QWK: 0.3969
Best val QWK so far: 0.4088
Val derived_overall_from_traits_qwk: 0.3995
[EARLY STOPPING] No improvement for 7/8 epochs.
--------------------------------------------------------------------------------


Epoch 26/80:   0%|          | 0/242 [00:00<?, ?it/s]


Epoch 26 train loss: 0.003787


Predict:   0%|          | 0/31 [00:00<?, ?it/s]


== val epoch 26 ==
TR_qwk: 0.3825
TR_mae: 1.1543
CC_qwk: 0.3715
CC_mae: 1.3091
LR_qwk: 0.3903
LR_mae: 1.2234
GRA_qwk: 0.4231
GRA_mae: 1.2595
mean_trait_qwk: 0.3918
mean_trait_mae: 1.2366
derived_overall_from_traits_qwk: 0.3923
derived_overall_from_traits_mae: 1.2384
Selected validation metric: mean_trait_qwk
Current val QWK: 0.3918
Best val QWK so far: 0.4088
Val derived_overall_from_traits_qwk: 0.3923
[EARLY STOPPING] No improvement for 8/8 epochs.
[EARLY STOPPING] Triggered.
Stopped at epoch 26.

Training finished.
Best validation QWK: 0.4088
Best checkpoint path: /content/ielts_lstm_essay_plus_qwen_discourse_trait_stacks/best_lstm_essay_plus_qwen_discourse.pt


In [17]:
# =========================
# LOAD BEST CHECKPOINT AND EVALUATE ON TEST SET
# =========================

checkpoint = torch.load(
    best_path,
    map_location=DEVICE,
    weights_only=False,   # needed for PyTorch >= 2.6 when checkpoint stores numpy/dict metadata
)

model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)

print("Loaded checkpoint from:", best_path)
print("Best epoch:", checkpoint.get("epoch"))
print("Best validation QWK:", checkpoint.get("best_val_qwk"))
print("Selected metric:", checkpoint.get("selected_metric_name"))

test_metrics, y_true_band, y_pred_band, overall_pred = evaluate_lstm(
    model,
    test_loader,
    name="test best LSTM essay+Qwen-style discourse",
)

Loaded checkpoint from: /content/ielts_lstm_essay_plus_qwen_discourse_trait_stacks/best_lstm_essay_plus_qwen_discourse.pt
Best epoch: 18
Best validation QWK: 0.4088125250897112
Selected metric: mean_trait_qwk


Predict:   0%|          | 0/31 [00:00<?, ?it/s]


== test best LSTM essay+Qwen-style discourse ==
TR_qwk: 0.3783
TR_mae: 1.1763
CC_qwk: 0.3638
CC_mae: 1.3036
LR_qwk: 0.4126
LR_mae: 1.2010
GRA_qwk: 0.4244
GRA_mae: 1.2619
mean_trait_qwk: 0.3948
mean_trait_mae: 1.2357
derived_overall_from_traits_qwk: 0.3940
derived_overall_from_traits_mae: 1.2387


In [18]:
# =========================
# EXPORT TEST PREDICTIONS
# =========================

def build_prediction_df_lstm(df, y_true_band, y_pred_band, overall_pred):
    out = df.reset_index(drop=True).copy()

    for i, trait in enumerate(TRAIT_COLS):
        out[f"gold_{trait}"] = y_true_band[:, i]
        out[f"pred_{trait}"] = y_pred_band[:, i]
        out[f"abs_error_{trait}"] = (
            out[f"pred_{trait}"] - out[f"gold_{trait}"]
        ).abs()

    out["pred_Overall_derived"] = overall_pred

    if "Overall" in out.columns:
        out["gold_Overall"] = out["Overall"]

        valid = ~pd.isna(out["gold_Overall"])

        if valid.any():
            print(
                "Derived Overall QWK:",
                qwk_band(
                    out.loc[valid, "gold_Overall"],
                    out.loc[valid, "pred_Overall_derived"],
                ),
            )

            print(
                "Derived Overall MAE:",
                mean_absolute_error(
                    out.loc[valid, "gold_Overall"],
                    out.loc[valid, "pred_Overall_derived"],
                ),
            )

    out["mean_abs_trait_error"] = out[
        [f"abs_error_{t}" for t in TRAIT_COLS]
    ].mean(axis=1)

    return out


pred_df_lstm = build_prediction_df_lstm(
    test_df,
    y_true_band,
    y_pred_band,
    overall_pred,
)

pred_path = os.path.join(
    RNN_OUTPUT_DIR,
    "test_predictions_lstm_essay_plus_qwen_discourse_trait_stacks.csv",
)

pred_df_lstm.to_csv(pred_path, index=False)

print("Saved:", pred_path)
display(pred_df_lstm.head())


Saved: /content/ielts_lstm_essay_plus_qwen_discourse_trait_stacks/test_predictions_lstm_essay_plus_qwen_discourse_trait_stacks.csv


,prompt,essay,evaluation,essay_len,TR,CC,LR,GRA,n_found,parse_quality,...,pred_CC,abs_error_CC,gold_LR,pred_LR,abs_error_LR,gold_GRA,pred_GRA,abs_error_GRA,pred_Overall_derived,mean_abs_trait_error
0,18.Education of young people is highly priorit...,It can be argued that the vast majority of cou...,Task Achievement: \r\nThe essay adequately add...,303,7.0,7.0,6.0,6.0,4,good,...,6.5,0.5,6.0,6.0,0.0,6.0,6.0,0.0,6.0,0.250
1,Nowadays in many countries most shops and prod...,"In today's modern world, daily use things are ...",Task Achievement:\r\nThe candidate has adequat...,300,6.0,5.5,5.5,5.5,4,good,...,5.5,0.0,5.5,5.5,0.0,5.5,5.0,0.5,5.5,0.125
2,Some people believe that the rage of technolog...,Technology is deeply relative to the living of...,Task Achievement: 5.5\r\n- The candidate has e...,296,5.5,6.0,6.0,6.0,4,good,...,6.5,0.5,6.0,6.0,0.0,6.0,6.0,0.0,6.0,0.375
3,The world has many towns and cites constructed...,Numerous cities have houses and replanning and...,Task Achievement:\r\n\r\nThe essay adequately ...,342,6.5,6.5,6.0,6.0,4,good,...,8.5,2.0,6.0,7.5,1.5,6.0,8.0,2.0,8.0,1.750
4,Many people argue that in order to improve the...,"In this modern world, the importance of higher...",Task Achievement: (Band Score: 4)\r\n- The can...,264,4.0,3.5,3.0,3.0,4,good,...,3.5,0.0,3.0,3.5,0.5,3.0,3.5,0.5,3.5,0.250


In [19]:
# =========================
# EXPORT METRIC SUMMARY
# =========================

summary_rows = []

for trait in TRAIT_COLS:
    qwk = qwk_band(
        pred_df_lstm[f"gold_{trait}"],
        pred_df_lstm[f"pred_{trait}"],
    )

    mae = mean_absolute_error(
        pred_df_lstm[f"gold_{trait}"],
        pred_df_lstm[f"pred_{trait}"],
    )

    summary_rows.append({
        "model": "BiLSTM essay+Qwen-style discourse",
        "target": trait,
        "QWK": qwk,
        "MAE": mae,
    })

# Derived overall from gold traits
gold_overall_from_traits = np.mean(
    pred_df_lstm[[f"gold_{t}" for t in TRAIT_COLS]].values,
    axis=1,
)
gold_overall_from_traits = np.round(gold_overall_from_traits * 2) / 2

summary_rows.append({
    "model": "BiLSTM essay+Qwen-style discourse",
    "target": "Overall_derived_from_traits",
    "QWK": qwk_band(gold_overall_from_traits, pred_df_lstm["pred_Overall_derived"]),
    "MAE": mean_absolute_error(gold_overall_from_traits, pred_df_lstm["pred_Overall_derived"]),
})

# Derived overall compared with real Overall column if available
if "gold_Overall" in pred_df_lstm.columns:
    valid = ~pd.isna(pred_df_lstm["gold_Overall"])

    if valid.any():
        qwk = qwk_band(
            pred_df_lstm.loc[valid, "gold_Overall"],
            pred_df_lstm.loc[valid, "pred_Overall_derived"],
        )

        mae = mean_absolute_error(
            pred_df_lstm.loc[valid, "gold_Overall"],
            pred_df_lstm.loc[valid, "pred_Overall_derived"],
        )

        summary_rows.append({
            "model": "BiLSTM essay+Qwen-style discourse",
            "target": "Overall_derived",
            "QWK": qwk,
            "MAE": mae,
        })

summary_df_lstm = pd.DataFrame(summary_rows)

summary_path = os.path.join(
    RNN_OUTPUT_DIR,
    "test_metric_summary_lstm_essay_plus_qwen_discourse.csv",
)

summary_df_lstm.to_csv(summary_path, index=False)

print("Saved:", summary_path)
display(summary_df_lstm)


Saved: /content/ielts_lstm_essay_plus_qwen_discourse_trait_stacks/test_metric_summary_lstm_essay_plus_qwen_discourse.csv


,model,target,QWK,MAE
0,BiLSTM essay+Qwen-style discourse,TR,0.378310,1.176289
1,BiLSTM essay+Qwen-style discourse,CC,0.363794,1.303608
2,BiLSTM essay+Qwen-style discourse,LR,0.412645,1.201031
3,BiLSTM essay+Qwen-style discourse,GRA,0.424420,1.261856
4,BiLSTM essay+Qwen-style discourse,Overall_derived_from_traits,0.393982,1.238660


In [20]:
# =========================
# QUICK LEAKAGE CHECKS
# These checks should ideally show zero essay overlap across splits.
# =========================

train_essays = set(train_df["essay"].astype(str))
val_essays = set(val_df["essay"].astype(str))
test_essays = set(test_df["essay"].astype(str))

print("Train-Val overlap:", len(train_essays & val_essays))
print("Train-Test overlap:", len(train_essays & test_essays))
print("Val-Test overlap:", len(val_essays & test_essays))

print("\nThis notebook does NOT auto-detect numeric CSV columns as discourse features.")
print("Discourse features are extracted from prompt + essay only.")


Train-Val overlap: 0
Train-Test overlap: 1
Val-Test overlap: 0

This notebook does NOT auto-detect numeric CSV columns as discourse features.
Discourse features are extracted from prompt + essay only.
